# 面试问题：不用 `torch.optim`，怎样从零实现 SGD、Momentum、Adam 与 AdamW？

**一句话回答**：优化器维护的是“参数名到状态”的显式状态机。SGD 沿当前梯度走；Momentum 累积指数移动平均；Adam 同时估计一阶、二阶矩并对初始化偏差做修正；AdamW 把权重衰减从损失梯度中解耦。工程上还必须处理参数组、禁止衰减的 bias/Norm、状态序列化、梯度裁剪顺序和断点恢复。

本 Notebook 只用 NumPy 手写 update rule，并用有限差分、解析首步、训练曲线与逐位一致的断点恢复验证实现。

In [ ]:
import copy, hashlib, json, math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED97=9701; rng97=np.random.default_rng(SEED97)  # 计算并保存当前步骤的中间状态。
x97=rng97.normal(size=(160,4)); true_w97=np.array([1.5,-2.,.7,3.]); y97=x97@true_w97+.35+rng97.normal(0,.05,160)  # 计算并保存当前步骤的中间状态。
assert x97.shape==(160,4) and y97.shape==(160,)  # 用受控断言验证关键不变量。
assert np.linalg.matrix_rank(x97)==4  # 用受控断言验证关键不变量。
assert SEED97==9701  # 用受控断言验证关键不变量。

## 1. 先定义参数、目标函数与梯度合同

优化器不应该知道模型结构，只接收同名参数和梯度；模型负责求梯度。下面的均方误差采用 `mean` reduction，因此 batch 大小变化不会偷偷改变学习率。有限差分只做小规模验算，生产训练用解析梯度或 autograd。

In [ ]:
def loss_grad97(params,x=x97,y=y97):  # 定义本节可复用的核心函数。
    pred=x@params["weight"]+params["bias"]; err=pred-y  # 计算并保存当前步骤的中间状态。
    return float(np.mean(err**2)),{"weight":2*x.T@err/len(x),"bias":np.array(2*err.mean())}  # 返回当前分支计算出的结果。
p97={"weight":rng97.normal(size=4),"bias":np.array(0.)}; loss97,g97=loss_grad97(p97)  # 计算并保存当前步骤的中间状态。
eps97=1e-6; plus97=copy.deepcopy(p97); minus97=copy.deepcopy(p97); plus97["weight"][0]+=eps97; minus97["weight"][0]-=eps97  # 计算并保存当前步骤的中间状态。
numeric97=(loss_grad97(plus97)[0]-loss_grad97(minus97)[0])/(2*eps97)  # 计算并保存当前步骤的中间状态。
assert math.isclose(numeric97,g97["weight"][0],rel_tol=1e-6)  # 用受控断言验证关键不变量。
assert g97["weight"].shape==p97["weight"].shape and g97["bias"].shape==p97["bias"].shape  # 用受控断言验证关键不变量。
assert loss97>0 and all(np.isfinite(v).all() for v in g97.values())  # 用受控断言验证关键不变量。

## 2. SGD：最小但完整的基线

更新式是 `θ ← θ - lr·g`。实现仍要校验键、shape 和有限值，避免某层漏传梯度或广播后悄悄更新错误。SGD 没有历史状态，适合作为排查复杂优化器的基线。

In [ ]:
class SGD97:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,lr):  # 定义本节可复用的核心函数。
        if lr<=0: raise ValueError("lr_contract")  # 按当前条件选择后续控制路径。
        self.lr=float(lr)  # 计算并保存当前步骤的中间状态。
    def step(self,params,grads):  # 定义本节可复用的核心函数。
        if params.keys()!=grads.keys(): raise ValueError("key_contract")  # 按当前条件选择后续控制路径。
        for k in params:  # 遍历输入元素以累积或检查结果。
            if params[k].shape!=grads[k].shape or not np.isfinite(grads[k]).all(): raise ValueError("grad_contract")  # 按当前条件选择后续控制路径。
            params[k]-=self.lr*grads[k]  # 计算并保存当前步骤的中间状态。
sgd_p97={"weight":np.zeros(4),"bias":np.array(0.)}; sgd97=SGD97(.08); before97=loss_grad97(sgd_p97)[0]  # 计算并保存当前步骤的中间状态。
for _ in range(80): _,gg97=loss_grad97(sgd_p97); sgd97.step(sgd_p97,gg97)  # 遍历输入元素以累积或检查结果。
after97=loss_grad97(sgd_p97)[0]  # 计算并保存当前步骤的中间状态。
assert after97<before97*.01  # 用受控断言验证关键不变量。
assert np.linalg.norm(sgd_p97["weight"]-true_w97)<.1  # 用受控断言验证关键不变量。
try: SGD97(0); raise AssertionError("bad lr accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="lr_contract"  # 捕获预期异常并验证失败分支。

## 3. Momentum：状态的时序不能写反

这里采用 `v_t=βv_{t-1}+g_t, θ_t=θ_{t-1}-lr·v_t`。有些资料把 `(1-β)` 放进梯度，二者只是尺度约定，但学习率不可直接混用。面试时应先声明公式，再解释速度状态必须和参数同 shape、同名持久化。

In [ ]:
class Momentum97:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,lr,beta=.9): self.lr,self.beta=float(lr),float(beta); self.velocity={}  # 定义本节可复用的核心函数。
    def step(self,params,grads):  # 定义本节可复用的核心函数。
        for k,p in params.items():  # 遍历输入元素以累积或检查结果。
            v=self.velocity.setdefault(k,np.zeros_like(p)); v*=self.beta; v+=grads[k]; p-=self.lr*v  # 计算并保存当前步骤的中间状态。
    def state_dict(self): return {"lr":self.lr,"beta":self.beta,"velocity":copy.deepcopy(self.velocity)}  # 定义本节可复用的核心函数。
q97={"x":np.array(1.)}; mom97=Momentum97(.1,.9); mom97.step(q97,{"x":np.array(1.)}); first97=float(q97["x"]); mom97.step(q97,{"x":np.array(1.)})  # 计算并保存当前步骤的中间状态。
assert math.isclose(first97,.9) and math.isclose(float(q97["x"]),.71)  # 用受控断言验证关键不变量。
assert math.isclose(float(mom97.velocity["x"]),1.9)  # 用受控断言验证关键不变量。
assert mom97.state_dict()["velocity"]["x"] is not mom97.velocity["x"]  # 用受控断言验证关键不变量。

## 4. Adam：两组矩估计与偏差修正

零初始化让早期 `m_t、v_t` 系统性偏小，所以使用 `m/(1-β₁ᵗ)` 与 `v/(1-β₂ᵗ)`。第一步经修正后应近似 `lr·sign(g)`；漏掉修正是常见手写错误。`epsilon` 放在平方根外是这里的明确合同。

In [ ]:
class Adam97:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,lr=.001,beta1=.9,beta2=.999,eps=1e-8):  # 定义本节可复用的核心函数。
        self.lr,self.beta1,self.beta2,self.eps=float(lr),float(beta1),float(beta2),float(eps); self.t=0; self.m={}; self.v={}  # 计算并保存当前步骤的中间状态。
    def step(self,params,grads):  # 定义本节可复用的核心函数。
        self.t+=1  # 计算并保存当前步骤的中间状态。
        for k,p in params.items():  # 遍历输入元素以累积或检查结果。
            self.m.setdefault(k,np.zeros_like(p)); self.v.setdefault(k,np.zeros_like(p)); g=grads[k]  # 计算并保存当前步骤的中间状态。
            self.m[k]=self.beta1*self.m[k]+(1-self.beta1)*g; self.v[k]=self.beta2*self.v[k]+(1-self.beta2)*g*g  # 计算并保存当前步骤的中间状态。
            mh=self.m[k]/(1-self.beta1**self.t); vh=self.v[k]/(1-self.beta2**self.t); p-=self.lr*mh/(np.sqrt(vh)+self.eps)  # 计算并保存当前步骤的中间状态。
    def state_dict(self): return {"t":self.t,"m":copy.deepcopy(self.m),"v":copy.deepcopy(self.v),"lr":self.lr,"beta1":self.beta1,"beta2":self.beta2,"eps":self.eps}  # 定义本节可复用的核心函数。
    def load_state_dict(self,s): self.t=s["t"]; self.m=copy.deepcopy(s["m"]); self.v=copy.deepcopy(s["v"])  # 定义本节可复用的核心函数。
aparam97={"x":np.array([1.,1.])}; adam97=Adam97(lr=.01); adam97.step(aparam97,{"x":np.array([2.,-3.])})  # 计算并保存当前步骤的中间状态。
assert np.allclose(aparam97["x"],[.99,1.01],atol=1e-9)  # 用受控断言验证关键不变量。
assert adam97.t==1 and np.all(adam97.v["x"]>0)  # 用受控断言验证关键不变量。
assert set(adam97.state_dict())=={"t","m","v","lr","beta1","beta2","eps"}  # 用受控断言验证关键不变量。

## 5. Adam + L2 不等于 AdamW

把 `λθ` 加进 Adam 梯度会被二阶矩归一化，衰减量依赖梯度历史；AdamW 则独立执行 `θ←(1-lr·λ)θ`，语义是稳定的乘法收缩。通常不对 bias、LayerNorm/RMSNorm 的一维 scale 做衰减。

In [ ]:
coupled97={"x":np.array(2.)}; ac97=Adam97(lr=.01); ac97.step(coupled97,{"x":np.array(.2)})  # 计算并保存当前步骤的中间状态。
decoupled97=np.array(2.); decoupled97*=1-.01*.1  # 计算并保存当前步骤的中间状态。
assert math.isclose(float(coupled97["x"]),1.99,rel_tol=1e-8)  # 用受控断言验证关键不变量。
assert math.isclose(float(decoupled97),1.998,rel_tol=1e-12)  # 用受控断言验证关键不变量。
assert not math.isclose(float(coupled97["x"]),float(decoupled97))  # 用受控断言验证关键不变量。

## 6. 参数组：名字规则必须可审计

常见规则是矩阵/卷积核衰减，一维向量与 bias 不衰减。仅凭名称容易漏掉自定义 Norm，仅凭维度也可能误判 embedding；生产代码应由模型显式标注，并在启动时打印清单与总参数量。

In [ ]:
named97={"encoder.weight":np.ones((3,4)),"encoder.bias":np.ones(3),"norm.weight":np.ones(3),"head.weight":np.ones((2,3))}  # 计算并保存当前步骤的中间状态。
def partition97(named):  # 定义本节可复用的核心函数。
    decay=[]; no_decay=[]  # 计算并保存当前步骤的中间状态。
    for name,p in named.items(): (no_decay if p.ndim<2 or name.endswith("bias") else decay).append(name)  # 遍历输入元素以累积或检查结果。
    return sorted(decay),sorted(no_decay)  # 返回当前分支计算出的结果。
decay97,no_decay97=partition97(named97)  # 计算并保存当前步骤的中间状态。
before_group97={k:v.copy() for k,v in named97.items()}  # 计算并保存当前步骤的中间状态。
for k in decay97: named97[k]*=1-.1*.01  # 遍历输入元素以累积或检查结果。
assert decay97==["encoder.weight","head.weight"]  # 用受控断言验证关键不变量。
assert no_decay97==["encoder.bias","norm.weight"]  # 用受控断言验证关键不变量。
assert np.all(named97["encoder.weight"]<before_group97["encoder.weight"]) and np.array_equal(named97["norm.weight"],before_group97["norm.weight"])  # 用受控断言验证关键不变量。

## 7. 在同一任务上比较收敛，而不是背口号

优化器优劣依赖曲率、噪声、batch、调度器和预算。公平比较要使用相同初始化、数据顺序与更新次数，并分别调学习率。这里只验证两种实现都能把线性回归损失显著降下来，不声称 Adam 永远更好。

In [ ]:
init97={"weight":np.zeros(4),"bias":np.array(0.)}; pa97=copy.deepcopy(init97); ps97=copy.deepcopy(init97); oa97=Adam97(.08); os97=SGD97(.08); hist_a97=[]; hist_s97=[]  # 计算并保存当前步骤的中间状态。
for _ in range(70):  # 遍历输入元素以累积或检查结果。
    la,ga=loss_grad97(pa97); ls,gs=loss_grad97(ps97); hist_a97.append(la); hist_s97.append(ls); oa97.step(pa97,ga); os97.step(ps97,gs)  # 计算并保存当前步骤的中间状态。
assert hist_a97[-1]<hist_a97[0]*.01 and hist_s97[-1]<hist_s97[0]*.01  # 用受控断言验证关键不变量。
assert np.linalg.norm(pa97["weight"]-true_w97)<.15  # 用受控断言验证关键不变量。
assert np.isfinite(hist_a97).all() and np.isfinite(hist_s97).all()  # 用受控断言验证关键不变量。

## 8. 断点恢复必须包含优化器状态

只加载模型权重会丢失 Adam 的 `t/m/v`，恢复后的下一步与原训练不一致。下面在第 11 步保存参数与状态，分别连续和恢复执行后续梯度，要求结果逐位相同；这也是优化器实现最有价值的回归测试。

In [ ]:
p_run97=copy.deepcopy(init97); o_run97=Adam97(.03)  # 计算并保存当前步骤的中间状态。
for _ in range(11): _,g=loss_grad97(p_run97); o_run97.step(p_run97,g)  # 遍历输入元素以累积或检查结果。
checkpoint97={"params":copy.deepcopy(p_run97),"optimizer":o_run97.state_dict()}  # 计算并保存当前步骤的中间状态。
for _ in range(9): _,g=loss_grad97(p_run97); o_run97.step(p_run97,g)  # 遍历输入元素以累积或检查结果。
p_resume97=copy.deepcopy(checkpoint97["params"]); o_resume97=Adam97(.03); o_resume97.load_state_dict(checkpoint97["optimizer"])  # 计算并保存当前步骤的中间状态。
for _ in range(9): _,g=loss_grad97(p_resume97); o_resume97.step(p_resume97,g)  # 遍历输入元素以累积或检查结果。
manifest97={"optimizer":"AdamW-style-ready","step":o_resume97.t,"state_keys":["m","v"],"reduction":"mean"}; digest97=hashlib.sha256(json.dumps(manifest97,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert all(np.array_equal(p_run97[k],p_resume97[k]) for k in p_run97)  # 用受控断言验证关键不变量。
assert o_resume97.t==20 and all(np.array_equal(o_run97.m[k],o_resume97.m[k]) for k in o_run97.m)  # 用受控断言验证关键不变量。
assert len(digest97)==64 and manifest97["reduction"]=="mean"  # 用受控断言验证关键不变量。

## 面试总结

一个完整回答应按 **统一梯度合同 → SGD 基线 → Momentum 状态 → Adam 偏差修正 → AdamW 解耦衰减 → 参数组 → 公平实验 → 完整 checkpoint** 展开。追问 epsilon、L2 与 weight decay、为何 Norm 不衰减、恢复时漏掉 `t` 会怎样，都能落回这里的公式和测试。

延伸阅读：[PyTorch Optimizer 语义](https://pytorch.org/docs/stable/optim.html)、[Adam 论文](https://arxiv.org/abs/1412.6980)、[Decoupled Weight Decay](https://arxiv.org/abs/1711.05101)。